In [1]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [6]:
v1.dot(dv) #we compare the query against the document 

np.float32(0.3233239)

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [8]:
v2.dot(dv)

np.float32(0.019730505)

In [9]:
from ingest import load_faq_data

documents = load_faq_data()

In [10]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [11]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
len(texts)

1380

In [13]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1380

In [14]:
vectors[0].shape

(384,)

In [15]:
scores = []
for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [16]:
import numpy as np
x=np.array(vectors)

In [17]:
x

array([[-1.03578933e-01, -6.19631819e-02, -1.13746086e-02, ...,
         5.81632666e-02, -2.28136890e-02, -1.11605851e-02],
       [-1.16193146e-01, -1.01877376e-01,  5.09268939e-02, ...,
        -3.24033462e-02,  2.07024459e-02,  5.45929223e-02],
       [-7.64710680e-02,  6.32362717e-05,  3.17689702e-02, ...,
         1.90290753e-02,  8.44087675e-02, -3.48371379e-02],
       ...,
       [-4.17902656e-02, -1.71949975e-02, -4.15087305e-02, ...,
        -5.01084998e-02, -6.99306419e-03, -6.95181731e-03],
       [ 7.29597453e-03, -2.02237200e-02, -9.10493210e-02, ...,
         7.81215131e-02,  4.06462401e-02, -8.07823148e-03],
       [ 5.98530807e-02,  4.52837311e-02, -4.05396298e-02, ...,
        -3.24840918e-02, -3.59650552e-02, -1.35418130e-02]],
      shape=(1380, 384), dtype=float32)

In [18]:
scores = x.dot(v1)

In [19]:
scores  

array([ 0.23396152,  0.05057271,  0.0056146 , ...,  0.09541064,
       -0.04206593,  0.03490692], shape=(1380,), dtype=float32)

In [20]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(860), np.float32(0.7629409))

In [21]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [22]:
top5 = np.argsort(scores)[-5:]

In [23]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([ 860,  473,   29, 1262,  865])

In [24]:
scores[top5]

array([0.7629409 , 0.757937  , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

## vector search & min search

In [25]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(x, documents)

In [26]:
vindex.search(   v1,
 filter_dict={"course": "llm-zoomcamp"})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project s

## RAG with vectory seearch

In [27]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [41]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RagHelper


assistant = RagHelper(
    index=index,
    llm_client=openai_client,
    # instructions=INSTRUCTIONS,
    model="meta-llama/llama-3.1-8b-instruct:free",
)

In [ ]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

In [37]:
class RAGVector(RagHelper):

    def __init__(self, embedder,course='llm-zoomcamp', **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
        self.course = course

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [38]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
    model="nvidia/nemotron-3-ultra-550b-a55b:free",

)

In [39]:
vector_assistant.rag("the program has already begun, can I still sign up?")


RuntimeError: Empty response from model nvidia/nemotron-3-ultra-550b-a55b:free: ChatCompletion(id=None, choices=None, created=None, model=None, object=None, moderation=None, service_tier=None, system_fingerprint=None, usage=None, error={'message': 'Upstream error from Nvidia: ResourceExhausted: Worker local total request limit reached (32/32)', 'code': 502})